# Neural networks for forecasting: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression

### 1. Sunspot numbers (yearly), included with statsmodels

In [ ]:
# 1. Sunspot numbers (yearly), included with statsmodels
y = sm.datasets.sunspots.load_pandas().data["SUNACTIVITY"].values.astype(float)
n = len(y)
print("observations:", n)

def windows(z, p):
    X = np.array([z[t - p:t] for t in range(p, len(z))])
    return X, z[p:]

### 2. Scale with training statistics only, then build windows

In [ ]:
# 2. Scale with training statistics only, then build windows
split = int(n * 0.8)
mu, sd = y[:split].mean(), y[:split].std()
z = (y - mu) / sd
p = 9
X, t = windows(z, p)
Xtr, ytr, Xte, yte = X[:split - p], t[:split - p], X[split - p:], t[split - p:]
print("training windows:", len(ytr), "| test windows:", len(yte))

### 3. Linear AR(p) benchmark and a small neural network

In [ ]:
# 3. Linear AR(p) benchmark and a small neural network
ar = LinearRegression().fit(Xtr, ytr)
nn = MLPRegressor(hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs", alpha=1e-2, max_iter=3000, random_state=0).fit(Xtr, ytr)
rmse = lambda a, b: np.sqrt(np.mean((a - b) ** 2)) * sd
naive = Xte[:, -1]
print("\ntest RMSE: naive", round(rmse(naive, yte), 2), "| AR(9)", round(rmse(ar.predict(Xte), yte), 2), "| network", round(rmse(nn.predict(Xte), yte), 2))
nparam = sum(w.size for w in nn.coefs_) + sum(b.size for b in nn.intercepts_)
print("network parameters:", nparam, "= H(p+1) + H + 1 =", 8 * (p + 1) + 8 + 1)

### 4. Hidden units: training error falls, test error does not always follow

In [ ]:
# 4. Hidden units: training error falls, test error does not always follow
print("\nhidden units -> train RMSE, test RMSE")
for H in (1, 2, 4, 8, 16, 32):
    m = MLPRegressor(hidden_layer_sizes=(H,), activation="tanh", solver="lbfgs", alpha=1e-3, max_iter=4000, random_state=0).fit(Xtr, ytr)
    print(f"{H:3d}  {rmse(m.predict(Xtr), ytr):7.2f}  {rmse(m.predict(Xte), yte):7.2f}")

### 5. A chaotic series: the network can learn what a linear model cannot

In [ ]:
# 5. A chaotic series: the network can learn what a linear model cannot
rng = np.random.default_rng(1)
q = np.empty(600); q[0] = 0.3
for i in range(1, 600):
    q[i] = 3.85 * q[i - 1] * (1 - q[i - 1])
q += 0.01 * rng.standard_normal(600)
Xc, tc = windows(q, 2)
Xa, ya, Xb, yb = Xc[:400], tc[:400], Xc[400:], tc[400:]
lin = LinearRegression().fit(Xa, ya)
net = MLPRegressor(hidden_layer_sizes=(12,), activation="tanh", solver="lbfgs", max_iter=5000, random_state=0).fit(Xa, ya)
r2 = lambda m: 1 - np.sum((m.predict(Xb) - yb) ** 2) / np.sum((yb - yb.mean()) ** 2)
rm = lambda m: np.sqrt(np.mean((m.predict(Xb) - yb) ** 2))
print("\nchaotic map, test RMSE: linear", round(rm(lin), 4), "| network", round(rm(net), 4), "(R2:", round(r2(lin), 3), "and", round(r2(net), 3), ")")

## Expected output

Running all cells above should print:

```
observations: 309
training windows: 238 | test windows: 62

test RMSE: naive 33.28 | AR(9) 19.44 | network 26.18
network parameters: 89 = H(p+1) + H + 1 = 89

hidden units -> train RMSE, test RMSE
  1    12.95    22.46
  2    10.87    25.51
  4     9.27    29.47
  8     6.46    27.44
 16     2.23    30.91
 32     0.18    35.33

chaotic map, test RMSE: linear 0.0468 | network 0.0231 (R2: 0.98 and 0.995 )
```

## Practice exercises

Try these in the cells below.

1. Use the "Non-linear (threshold AR)" series. Compare the test RMSE of the network and of the linear AR model.
2. Increase hidden units to 20 and epochs to 1000. Watch the training and test curves separate (overfitting).
3. On the "Linear AR(2)" series the network cannot beat AR. Why?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
